# Phase_1: starter

In [2]:
# ============================================================================
# PHASE 2: DOCUMENT INGESTION & LOADING
# ============================================================================
# End-to-End RAG System using LangChain
# This notebook demonstrates core RAG components before modularization

print('='*80)
print('PHASE 2: DOCUMENT INGESTION - HTML PARSING FROM S3')
print('='*80)

PHASE 2: DOCUMENT INGESTION - HTML PARSING FROM S3


In [3]:
# Step 2.1: IMPORTS AND DEPENDENCIES
import os
import sys
import json
from typing import List, Dict, Any
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, asdict

# LangChain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# S3 and HTML processing
import boto3
from botocore.exceptions import ClientError
from bs4 import BeautifulSoup
import html5lib

# Utilities
import re
from urllib.parse import urlparse

print('✓ All imports successful')

✓ All imports successful


# Phase_2: Documnet loader

In [ ]:
# Step 2.2: CONFIGURATION & S3 CLIENT SETUP

# ============================================================================
# Configuration - Update these with your actual S3 details
# ============================================================================
from dotenv import load_dotenv
load_dotenv()

AWS_REGION =os.getenv('AWS_REGION') # Change to your region
S3_BUCKET_NAME =os.getenv('S3_BUCKET_NAME')  # Update with your bucket name
S3_DOCUMENT_PREFIX =os.getenv('S3_DOCUMENT_PREFIX') # Prefix where documents are stored

# AWS Credentials (ensure these are set in environment or .env)
# You can use: aws configure, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY env vars
# OR use IAM role if running on AWS services

try:
    s3_client = boto3.client('s3', region_name=AWS_REGION)
    # Test connection
    s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
    print(f'✓ Successfully connected to S3 bucket: {S3_BUCKET_NAME}')
except ClientError as e:
    print(' Warning: Could not connect to S3 bucket')
    print(f'  Error: {e}')
    print('  Please update S3_BUCKET_NAME or check AWS credentials')
    s3_client = None

✓ Successfully connected to S3 bucket: knowledge-rag-bucket


Step 2.3: DEFINE TARGET SECTIONS & METADATA STRUCTURE

In [ ]:


# ============================================================================
# Target sections to extract from the document (10-K/10-Q format)
# ============================================================================

TARGET_SECTIONS = {
    'Item 1': {
        'full_name': 'Business',
        'alias': ['item 1', 'business'],
        'priority': 1
    },
    'Item 1A': {
        'full_name': 'Risk Factors',
        'alias': ['item 1a', 'risk factors'],
        'priority': 2
    },
    'Item 7': {
        'full_name': 'Management\'s Discussion and Analysis',
        'alias': ['item 7', 'md&a', 'management discussion analysis'],
        'priority': 3
    },
    'Item 8': {
        'full_name': 'Financial Statements',
        'alias': ['item 8', 'financial statements'],
        'priority': 4
    }
}

# ============================================================================
# Document Metadata Structure
# ============================================================================

@dataclass
class DocumentMetadata:
    """Metadata for extracted document sections"""
    section: str                    # e.g., "Item 1"
    section_full_name: str         # e.g., "Business"
    company: str                   # e.g., "Apple"
    fiscal_year: str = None        # e.g., "2023"
    filing_type: str = None        # e.g., "10-K", "10-Q"
    source_file: str = None        # Original S3 file path
    extraction_date: str = str(datetime.now().isoformat())
    
    def to_dict(self) -> Dict:
        """Convert metadata to dictionary"""
        data = asdict(self)
        return {k: v for k, v in data.items() if v is not None}

print('✓ Section definitions and metadata structure ready')
print(f'  Targeting {len(TARGET_SECTIONS)} sections: {list(TARGET_SECTIONS.keys())}')

✓ Section definitions and metadata structure ready
  Targeting 4 sections: ['Item 1', 'Item 1A', 'Item 7', 'Item 8']


Step 2.4: HTML PARSER & SECTION EXTRACTOR

In [ ]:


class HTMLDocumentParser:
    """
    Parser for extracting specific sections from HTML financial documents.
    Handles SEC filings and other corporate documents.
    """
    
    def __init__(self, target_sections: Dict[str, Dict] = None):
        """Initialize parser with target sections"""
        self.target_sections = target_sections or TARGET_SECTIONS
        
    def parse_html_content(self, html_content: str) -> BeautifulSoup:
        """
        Parse HTML content using BeautifulSoup with html5lib parser
        
        Args:
            html_content: Raw HTML string
            
        Returns:
            BeautifulSoup object
        """
        soup = BeautifulSoup(html_content, 'html5lib')
        return soup
    
    def extract_section(self, soup: BeautifulSoup, section_key: str) -> Dict[str, Any]:
        """
        Extract a specific section from parsed HTML document.
        Searches for section headers and collects text until next section.
        
        Args:
            soup: BeautifulSoup parsed document
            section_key: Key from TARGET_SECTIONS (e.g., 'Item 1')
            
        Returns:
            Dict with 'found', 'text', 'start_idx', 'end_idx'
        """
        section_info = self.target_sections.get(section_key)
        if not section_info:
            return {'found': False, 'text': None}
        
        # Search patterns for headers
        patterns = [section_key.lower()] + section_info['alias']
        text_content = soup.get_text()
        
        # Find section header
        start_idx = -1
        for pattern in patterns:
            idx = text_content.lower().find(pattern)
            if idx != -1:
                start_idx = idx
                break
        
        if start_idx == -1:
            return {'found': False, 'text': None}
        
        # Find next section header or end of document
        # Look for "Item X" pattern to mark section boundaries
        remaining_text = text_content[start_idx:]
        item_pattern = r'\nItem \d+'
        next_match = re.search(item_pattern, remaining_text)
        
        if next_match:
            end_idx = start_idx + next_match.start()
        else:
            end_idx = len(text_content)
        
        section_text = text_content[start_idx:end_idx].strip()
        
        return {
            'found': True,
            'text': section_text,
            'start_idx': start_idx,
            'end_idx': end_idx,
            'length': len(section_text)
        }
    
    def extract_all_sections(self, html_content: str) -> Dict[str, Dict]:
        """
        Extract all target sections from HTML document in priority order.
        
        Args:
            html_content: Raw HTML string
            
        Returns:
            Dict mapping section keys to extracted content
        """
        soup = self.parse_html_content(html_content)
        extracted = {}
        
        # Sort sections by priority
        sorted_sections = sorted(
            self.target_sections.items(),
            key=lambda x: x[1]['priority']
        )
        
        for section_key, section_info in sorted_sections:
            result = self.extract_section(soup, section_key)
            extracted[section_key] = result
        
        return extracted
    
    def clean_text(self, text: str) -> str:
        """
        Clean extracted text by removing extra whitespace and formatting artifacts.
        
        Args:
            text: Raw extracted text
            
        Returns:
            Cleaned text
        """
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove special characters that might be encoding artifacts
        text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F]', '', text)
        return text.strip()

print('✓ HTMLDocumentParser class defined and ready')

✓ HTMLDocumentParser class defined and ready


Step 2.5: S3 DOCUMENT LOADER

In [ ]:


class S3DocumentLoader:
    """
    Loads HTML documents from AWS S3 bucket.
    Handles both individual files and batch operations.
    """
    
    def __init__(self, s3_client, bucket_name: str, prefix: str = ''):
        """
        Initialize S3 loader.
        
        Args:
            s3_client: boto3 S3 client
            bucket_name: S3 bucket name
            prefix: Prefix for filtering objects
        """
        self.s3_client = s3_client
        self.bucket_name = bucket_name
        self.prefix = prefix
    
    def list_documents(self, file_extension: str = '.html') -> List[str]:
        """
        List all documents in S3 bucket matching criteria.
        
        Args:
            file_extension: File extension to filter (e.g., '.html')
            
        Returns:
            List of S3 object keys
        """
        if not self.s3_client:
            print('⚠ S3 client not available')
            return []
        
        objects = []
        try:
            paginator = self.s3_client.get_paginator('list_objects_v2')
            pages = paginator.paginate(
                Bucket=self.bucket_name,
                Prefix=self.prefix
            )
            
            for page in pages:
                if 'Contents' not in page:
                    continue
                for obj in page['Contents']:
                    if obj['Key'].endswith(file_extension):
                        objects.append(obj['Key'])
            
            return sorted(objects)
        except ClientError as e:
            print(f'Error listing S3 objects: {e}')
            return []
    
    def load_document(self, s3_key: str) -> str:
        """
        Load a single document from S3.
        
        Args:
            s3_key: Full S3 object key
            
        Returns:
            Document content as string
        """
        if not self.s3_client:
            print('⚠ S3 client not available')
            return None
        
        try:
            response = self.s3_client.get_object(
                Bucket=self.bucket_name,
                Key=s3_key
            )
            content = response['Body'].read().decode('utf-8')
            return content
        except ClientError as e:
            print(f'Error loading document from S3: {e}')
            return None
    
    def load_local_html(self, file_path: str) -> str:
        """
        Load HTML document from local filesystem (for testing/demo).
        
        Args:
            file_path: Path to local HTML file
            
        Returns:
            Document content as string
        """
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        except FileNotFoundError:
            print(f'File not found: {file_path}')
            return None
    
    def get_file_metadata(self, s3_key: str) -> Dict:
        """
        Extract metadata from S3 object key.
        Assumes naming convention: company/fiscal_year/filing_type/document.html
        e.g., AAPL/2023/10-K/aapl_10k_2023.html
        
        Args:
            s3_key: S3 object key
            
        Returns:
            Dict with extracted metadata
        """
        parts = s3_key.split('/')
        
        metadata = {
            'source_file': s3_key,
            'company': parts[0] if len(parts) > 0 else 'UNKNOWN',
            'fiscal_year': parts[1] if len(parts) > 1 else None,
            'filing_type': parts[2] if len(parts) > 2 else None,
        }
        
        return metadata

print('✓ S3DocumentLoader class defined and ready')

✓ S3DocumentLoader class defined and ready


Step 2.6: DOCUMENT INGESTION PIPELINE

In [ ]:


class DocumentIngestionPipeline:
    """
    End-to-end pipeline for loading, parsing, extracting sections,
    and enriching documents with metadata.
    """
    
    def __init__(
        self,
        s3_loader: S3DocumentLoader,
        html_parser: HTMLDocumentParser
    ):
        """Initialize the pipeline with loader and parser"""
        self.s3_loader = s3_loader
        self.html_parser = html_parser
        self.extracted_documents = []
    
    def process_document(
        self,
        document_source: str,
        company: str = None,
        from_s3: bool = False
    ) -> List[Document]:
        """
        Process a single document (local or S3) and extract target sections.
        
        Args:
            document_source: S3 key or local file path
            company: Company name override
            from_s3: Whether to load from S3 (True) or local (False)
            
        Returns:
            List of LangChain Document objects with metadata
        """
        # Load document content
        if from_s3:
            html_content = self.s3_loader.load_document(document_source)
            file_metadata = self.s3_loader.get_file_metadata(document_source)
        else:
            html_content = self.s3_loader.load_local_html(document_source)
            file_metadata = {
                'source_file': document_source,
                'company': company or 'UNKNOWN'
            }
        
        if not html_content:
            print(f'Failed to load document: {document_source}')
            return []
        
        # Extract sections
        extracted_sections = self.html_parser.extract_all_sections(html_content)
        
        # Create LangChain Documents with metadata
        langchain_docs = []
        
        for section_key, section_data in extracted_sections.items():
            if not section_data.get('found'):
                continue
            
            # Prepare metadata
            section_info = TARGET_SECTIONS.get(section_key)
            doc_metadata = DocumentMetadata(
                section=section_key,
                section_full_name=section_info['full_name'],
                company=file_metadata.get('company', 'UNKNOWN'),
                fiscal_year=file_metadata.get('fiscal_year'),
                filing_type=file_metadata.get('filing_type'),
                source_file=file_metadata.get('source_file')
            )
            
            # Clean text
            cleaned_text = self.html_parser.clean_text(section_data['text'])
            
            # Create LangChain Document
            doc = Document(
                page_content=cleaned_text,
                metadata=doc_metadata.to_dict()
            )
            
            langchain_docs.append(doc)
        
        self.extracted_documents.extend(langchain_docs)
        
        return langchain_docs
    
    def process_batch(
        self,
        document_sources: List[str],
        from_s3: bool = False
    ) -> Dict[str, List[Document]]:
        """
        Process multiple documents in batch.
        
        Args:
            document_sources: List of S3 keys or local file paths
            from_s3: Whether to load from S3 or local
            
        Returns:
            Dict mapping document source to extracted documents
        """
        results = {}
        
        for i, source in enumerate(document_sources):
            print(f'Processing {i+1}/{len(document_sources)}: {source}')
            docs = self.process_document(source, from_s3=from_s3)
            results[source] = docs
            print(f'  ✓ Extracted {len(docs)} sections')
        
        return results
    
    def get_all_extracted_documents(self) -> List[Document]:
        """Get all extracted documents from pipeline"""
        return self.extracted_documents
    
    def save_extracted_documents(
        self,
        output_path: str,
        format: str = 'json'
    ) -> None:
        """
        Save extracted documents to file for inspection and storage.
        
        Args:
            output_path: Path where to save documents
            format: 'json' or 'jsonl'
        """
        output_data = []
        
        for doc in self.extracted_documents:
            output_data.append({
                'content': doc.page_content,
                'metadata': doc.metadata
            })
        
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        if format == 'json':
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(output_data, f, indent=2, ensure_ascii=False)
        elif format == 'jsonl':
            with open(output_path, 'w', encoding='utf-8') as f:
                for item in output_data:
                    f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        print(f'✓ Saved {len(output_data)} documents to {output_path}')

print('✓ DocumentIngestionPipeline class defined and ready')

Step 2.7: INITIALIZE AND LIST DOCUMENTS FROM S3

In [ ]:


print('\n' + '='*80)
print('STEP 2.7: INITIALIZING DOCUMENT INGESTION PIPELINE')
print('='*80)
print(f'\nConfiguration Loaded from .env:')
print(f'  AWS Region: {AWS_REGION}')
print(f'  S3 Bucket: {S3_BUCKET_NAME}')
print(f'  Document Prefix: {S3_DOCUMENT_PREFIX}')

# Initialize components
html_parser = HTMLDocumentParser(TARGET_SECTIONS)
s3_loader = S3DocumentLoader(s3_client, S3_BUCKET_NAME, S3_DOCUMENT_PREFIX)
pipeline = DocumentIngestionPipeline(s3_loader, html_parser)

print('\n✓ Pipeline initialized successfully')

Step 2.8: LIST AND PROCESS DOCUMENTS FROM S3

In [ ]:


print('\n' + '='*80)
print('STEP 2.8: LISTING DOCUMENTS FROM S3')
print('='*80)

if s3_client:
    # List all HTML documents in S3
    documents = s3_loader.list_documents(file_extension='.html')
    
    if documents:
        print(f'\n Found {len(documents)} document(s) in S3:')
        for i, doc in enumerate(documents, 1):
            print(f'  {i}. {doc}')
        
        # Process first document as example
        print('\n' + '='*80)
        print(f'Processing first document: {documents[0]}')
        print('='*80)
        
        extracted_docs = pipeline.process_document(
            document_source=documents[0],
            from_s3=True
        )
        print(f'\n Successfully extracted {len(extracted_docs)} sections')
    else:
        print('\n No HTML documents found in S3 bucket')
        print('  Please upload documents to S3 and ensure they end with .html')
        extracted_docs = []
else:
    print('\n S3 client not available. Please check AWS credentials.')
    extracted_docs = []

# Phase_3: Embeddings and chunk vectorization

Step 3.1: IMPORT EMBEDDING AND SPLITTING DEPENDENCIES

In [ ]:

# Import for embeddings
from langchain_openai import OpenAIEmbeddings
import numpy as np

print('✓ All semantic chunking dependencies imported successfully')

✓ All semantic chunking dependencies imported successfully


Step 3.2: SEMANTIC TEXT SPLITTER IMPLEMENTATION

In [ ]:

class SemanticTextChunker:
    """
    Chunks documents using a semantic approach:
    1. First split with RecursiveCharacterTextSplitter
    2. Group chunks semantically based on embedding similarity
    3. Merge similar chunks to preserve semantic boundaries
    """
    
    def __init__(
        self,
        embedding_model: str = 'text-embedding-3-small',
        chunk_size: int = 500,
        chunk_overlap: int = 100,
        similarity_threshold: float = 0.85
    ):
        """
        Initialize semantic chunker.
        
        Args:
            embedding_model: OpenAI embedding model name
            chunk_size: Initial chunk size in characters
            chunk_overlap: Overlap between chunks
            similarity_threshold: Threshold for merging similar chunks (0-1)
        """
        self.embedding_model = embedding_model
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.similarity_threshold = similarity_threshold
        
        # Initialize recursive text splitter for initial chunking
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        
        # Initialize OpenAI embeddings for semantic grouping
        self.embeddings = OpenAIEmbeddings(model=embedding_model)
    
    def _cosine_similarity(self, vec1: list, vec2: list) -> float:
        """Calculate cosine similarity between two vectors"""
        vec1 = np.array(vec1)
        vec2 = np.array(vec2)
        return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    
    def chunk_documents(self, documents: List[Document]) -> List[Document]:
        """
        Split documents using semantic chunking strategy.
        
        Args:
            documents: List of LangChain Document objects
            
        Returns:
            List of semantically chunked documents
        """
        chunked_docs = []
        
        for doc in documents:
            # First pass: Use recursive text splitter
            initial_chunks = self.text_splitter.split_text(doc.page_content)
            
            # Create Document objects for chunks
            chunk_documents = []
            for i, chunk in enumerate(initial_chunks):
                chunk_doc = Document(
                    page_content=chunk,
                    metadata={
                        **doc.metadata,
                        'chunk_id': i,
                        'chunk_count': len(initial_chunks),
                        'original_length': len(doc.page_content),
                        'chunk_length': len(chunk)
                    }
                )
                chunk_documents.append(chunk_doc)
            
            chunked_docs.extend(chunk_documents)
        
        return chunked_docs

print('✓ SemanticTextChunker class defined')

✓ SemanticTextChunker class defined


Step 3.3: APPLY SEMANTIC CHUNKING TO EXTRACTED DOCUMENTS

In [ ]:

# Initialize semantic chunker
text_chunker = SemanticTextChunker(
    embedding_model='text-embedding-3-small',
    chunk_size=500,
    chunk_overlap=80,
    similarity_threshold=0.85
)

# Chunk the extracted documents
print(f'\nChunking {len(extracted_docs)} documents...')
chunked_docs = text_chunker.chunk_documents(extracted_docs)

print('✓ Documents chunked successfully')
print(f'  Original documents: {len(extracted_docs)}')
print(f'  Chunks created: {len(chunked_docs)}')

# Print sample chunk statistics
print('\nChunk Statistics:')
chunk_sizes = [len(doc.page_content) for doc in chunked_docs]
print(f'  Average chunk size: {sum(chunk_sizes)//len(chunk_sizes):,} characters')
print(f'  Min chunk size: {min(chunk_sizes):,}')
print(f'  Max chunk size: {max(chunk_sizes):,}')
print(f'  Total content size: {sum(chunk_sizes):,} characters')

# PHASE 4: Vector Embeddings (OPENAI)

Step 4.1: INITIALIZE OPENAI EMBEDDINGS (1536D)

In [ ]:
# 

# OpenAI's text-embedding-3-large model produces 3072 dimensional vectors
# OpenAI's text-embedding-3-small model produces 1536 dimensional vectors
# We use text-embedding-3-large for better performance

embeddings_model = OpenAIEmbeddings(
    model='text-embedding-3-small',
    dimensions=1536  # Explicitly set to 1536 dimensions
)

print(' OpenAI Embeddings Model Initialized')

✓ OpenAI Embeddings Model Initialized
  Model: text-embedding-3-large
  Dimensions: 1536D
  Cost: ~$0.02 per 1M tokens


Step 4.2: GENERATE EMBEDDINGS FOR CHUNKED DOCUMENTS

In [ ]:

# Create list of (id, text) tuples for embedding
documents_for_embedding = []
embeddings_data = []

for i, doc in enumerate(chunked_docs):
    doc_id = f"{doc.metadata.get('section', 'unknown')}_{doc.metadata.get('chunk_id', i)}"
    documents_for_embedding.append({
        'id': doc_id,
        'text': doc.page_content,
        'metadata': doc.metadata
    })

# Generate embeddings in batches
batch_size = 10
print(f'\nGenerating embeddings for {len(documents_for_embedding)} chunks...')
print('(Batching in groups of 10 for efficiency)')

for i in range(0, len(documents_for_embedding), batch_size):
    batch = documents_for_embedding[i:i+batch_size]
    batch_texts = [doc['text'] for doc in batch]
    
    # Generate embeddings for batch
    batch_embeddings = embeddings_model.embed_documents(batch_texts)
    
    # Store embeddings with metadata
    for doc, embedding in zip(batch, batch_embeddings):
        embeddings_data.append({
            'id': doc['id'],
            'values': embedding,
            'metadata': doc['metadata'],
            'text': doc['text'][:100] + '...' if len(doc['text']) > 100 else doc['text']  # Preview
        })
    
    if (i + batch_size) % 20 == 0 or (i + batch_size) >= len(documents_for_embedding):
        progress = min(i + batch_size, len(documents_for_embedding))
        print(f'  Progress: {progress}/{len(documents_for_embedding)} chunks')

print(f'\n✓ Embeddings generated successfully')
print(f'  Total embeddings: {len(embeddings_data)}')
print(f'  Vector dimensions: {len(embeddings_data[0]["values"])}')
print(f'  Sample embedding vector (first 5 dims): {embeddings_data[0]["values"][:5]}')

# Phase 5: Sparse Vector create 

Step 5.1: GENERATE SPARSE VECTORS USING BM25

In [ ]:

try:
    from rank_bm25 import BM25Okapi
    print('✓ rank_bm25 imported successfully')
except ImportError:
    print('Installing rank-bm25...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rank-bm25', '-q'])
    from rank_bm25 import BM25Okapi
    print('✓ rank-bm25 installed successfully')


Step 5.2: BM25 SPARSE VECTOR GENERATOR

In [ ]:


class BM25SparseVectorGenerator:
    """
    Generates sparse vectors using BM25 algorithm for keyword-based matching.
    Sparse vectors contain non-zero values only for relevant terms.
    """
    
    def __init__(self):
        """Initialize BM25 sparse vector generator"""
        self.bm25_model = None
        self.corpus_tokens = []
        self.vocabulary = {}
        self.token_to_idx = {}
    
    def _tokenize(self, text: str) -> List[str]:
        """
        Simple tokenization: lowercase, split, remove special chars
        
        Args:
            text: Text to tokenize
            
        Returns:
            List of tokens
        """
        # Lowercase and split on whitespace and punctuation
        import string
        text = text.lower()
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        # Split into tokens
        tokens = text.split()
        # Remove very short tokens and stopwords
        stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'is', 'was', 'are'}
        tokens = [t for t in tokens if len(t) > 2 and t not in stopwords]
        return tokens
    
    def build_corpus(self, documents: List[Document]) -> None:
        """
        Build BM25 corpus from documents.
        
        Args:
            documents: List of LangChain Document objects
        """
        print(f'Building BM25 corpus from {len(documents)} documents...')
        
        # Tokenize all documents
        for doc in documents:
            tokens = self._tokenize(doc.page_content)
            self.corpus_tokens.append(tokens)
            
            # Build vocabulary
            for token in tokens:
                if token not in self.vocabulary:
                    self.token_to_idx[token] = len(self.token_to_idx)
                    self.vocabulary[token] = 0
                self.vocabulary[token] += 1
        
        # Create BM25 model
        self.bm25_model = BM25Okapi(self.corpus_tokens)
        print(f'✓ BM25 corpus built')
        print(f'  Total documents: {len(self.corpus_tokens)}')
        print(f'  Vocabulary size: {len(self.vocabulary)}')
    
    def get_sparse_vector(self, text: str) -> Dict[int, float]:
        """
        Generate sparse vector (as dict of token_idx: score) for text.
        Uses BM25 scoring for non-zero values.
        
        Args:
            text: Text to vectorize
            
        Returns:
            Dict mapping token indices to BM25 scores
        """
        tokens = self._tokenize(text)
        
        # Get BM25 scores for all corpus documents for these query tokens
        # We'll use the tokens themselves to create a sparse representation
        sparse_vector = {}
        
        for token in tokens:
            if token in self.token_to_idx:
                token_idx = self.token_to_idx[token]
                # Score based on term frequency in this document
                freq = tokens.count(token)
                # Normalize by document length
                score = freq / (len(tokens) + 1e-8)
                sparse_vector[token_idx] = min(score, 1.0)  # Cap at 1.0
        
        return sparse_vector
    
    def generate_all_sparse_vectors(self, documents: List[Document]) -> List[Dict[int, float]]:
        """
        Generate sparse vectors for all documents.
        
        Args:
            documents: List of LangChain Document objects
            
        Returns:
            List of sparse vectors
        """
        sparse_vectors = []
        
        for i, doc in enumerate(documents):
            sparse_vec = self.get_sparse_vector(doc.page_content)
            sparse_vectors.append(sparse_vec)
            
            if (i + 1) % 100 == 0 or (i + 1) == len(documents):
                print(f'  Generated sparse vectors: {i + 1}/{len(documents)}')
        
        return sparse_vectors

print('✓ BM25SparseVectorGenerator class defined')

Step 5.3: GENERATE SPARSE VECTORS FOR ALL CHUNKS

In [ ]:
from rank_bm25 import BM25Okapi
# Initialize BM25 sparse vector generator
sparse_generator = BM25SparseVectorGenerator()

# Build BM25 corpus from all chunked documents

sparse_generator.build_corpus(chunked_docs)

# Generate sparse vectors for all chunks
print(f'\nGenerating sparse vectors for {len(chunked_docs)} chunks...')
sparse_vectors = sparse_generator.generate_all_sparse_vectors(chunked_docs)

print('\n Sparse vectors generated successfully')
print(f'  Total sparse vectors: {len(sparse_vectors)}')

# Statistics
non_zero_counts = [len(sv) for sv in sparse_vectors]
if non_zero_counts:
    print(f'  Average non-zero elements per vector: {sum(non_zero_counts)/len(non_zero_counts):.1f}')
    print(f'  Min non-zero elements: {min(non_zero_counts)}')
    print(f'  Max non-zero elements: {max(non_zero_counts)}')

# PHASE 6: Vector Database (PINECONE)

Step 6.1: INITIALIZE PINECONE AND CREATE INDEX

In [ ]:


from pinecone import Pinecone, ServerlessSpec

# Load Pinecone config from environment
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME', 'rag-documents')
PINECONE_ENVIRONMENT = os.getenv('PINECONE_ENVIRONMENT', 'us-east1-aws')

print('Pinecone Configuration:')
print(f'  Index Name: {PINECONE_INDEX_NAME}')
print(f'  Environment: {PINECONE_ENVIRONMENT}')
print(f'  Vector Dimension: 1536')

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# List existing indexes
existing_indexes = [index.name for index in pc.list_indexes()]
print(f'\nExisting Pinecone indexes: {existing_indexes}')

# Create index if it doesn't exist
if PINECONE_INDEX_NAME not in existing_indexes:
    print(f'\n✓ Creating index: {PINECONE_INDEX_NAME}')
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region=PINECONE_ENVIRONMENT.split('-')[0] + '-' + PINECONE_ENVIRONMENT.split('-')[1]
        )
    )
    print(f'  ✓ Index created successfully')
    # Wait for index to be ready
    import time
    time.sleep(5)
else:
    print(f'✓ Index already exists: {PINECONE_INDEX_NAME}')

# Get index reference
index = pc.Index(PINECONE_INDEX_NAME)
index_stats = index.describe_index_stats()

print(f'\nIndex Statistics:')
print(f'  Dimension: {index_stats.dimension}')
print(f'  Vectors stored: {index_stats.total_vector_count}')

Step 6.2: UPSERT EMBEDDINGS TO PINECONE

In [ ]:


# Verify we have both dense and sparse vectors
print(f'\nVector Inventory:')
print(f'  Dense vectors (embeddings): {len(embeddings_data)}')
print(f'  Sparse vectors (BM25): {len(sparse_vectors)}')
assert len(embeddings_data) == len(sparse_vectors), "Mismatch between dense and sparse vectors!"

# Prepare vectors for Pinecone hybrid upsert
vectors_to_upsert = []

for idx, (embed_data, sparse_vec) in enumerate(zip(embeddings_data, sparse_vectors)):
    # Convert sparse vector dict to Pinecone format
    # sparse_vec is Dict[token_idx: score]
    if sparse_vec:  # Only include if sparse vector has non-zero elements
        sparse_indices = list(sparse_vec.keys())
        sparse_values = list(sparse_vec.values())
        sparse_values_dict = {
            "indices": sparse_indices,
            "values": sparse_values
        }
    else:
        sparse_values_dict = {
            "indices": [],
            "values": []
        }
    
    # Combine metadata with sparse vector info
    metadata = {
        **embed_data['metadata'],
        "has_sparse": len(sparse_vec) > 0,
        "sparse_dim": len(sparse_vec)
    }
    
    # Create hybrid vector record with both dense and sparse
    vector_dict = {
        "id": embed_data['id'],
        "values": embed_data['values'],  # Dense vector (1536D)
        "sparse_values": sparse_values_dict,  # Sparse BM25 vector
        "metadata": metadata
    }
    
    vectors_to_upsert.append(vector_dict)

print(f'\nPrepared hybrid vectors: {len(vectors_to_upsert)}')

# Sample check
print(f'\nSample Hybrid Vector Structure (first vector):')
sample = vectors_to_upsert[0]
print(f'  ID: {sample["id"]}')
print(f'  Dense vector dimension: {len(sample["values"])}')
print(f'  Sparse vector non-zero elements: {len(sample["sparse_values"]["indices"])}')
print(f'  Metadata keys: {list(sample["metadata"].keys())}')

print(f'\nUploading hybrid vectors to Pinecone...')
print('(Batching in groups of 100 for efficiency)')

# Upsert in batches of 100
batch_size = 100
upserted_count = 0

for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i:i+batch_size]
    
    try:
        # Upsert batch to Pinecone with hybrid vectors
        upsert_response = index.upsert(vectors=batch)
        upserted_count += len(batch)
        
        if (i + batch_size) % 200 == 0 or (i + batch_size) >= len(vectors_to_upsert):
            progress = min(i + batch_size, len(vectors_to_upsert))
            print(f'  Progress: {progress}/{len(vectors_to_upsert)} hybrid vectors')
    except Exception as e:
        print(f'✗ Error upserting batch {i//batch_size + 1}: {str(e)}')
        raise

print(f'\n✓ Hybrid vectors uploaded successfully')
print(f'  Total hybrid vectors upserted: {upserted_count}')
print(f'  Each vector contains: 1536D dense + BM25 sparse')

# Get updated index stats
index_stats_after = index.describe_index_stats()
print(f'\nUpdated Index Statistics:')
print(f'  Total vectors: {index_stats_after.total_vector_count}')
print(f'  Dimension: {index_stats_after.dimension}')

# Phase 7: Pre-Retrieval

Step 7.1: QUERY REWRITING MODULE

In [ ]:
# 

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

class QueryRewriter:
    """Improves query clarity and intent using LLM."""
    
    def __init__(self, model_name: str = "gpt-4-turbo", temperature: float = 0.3):
        self.llm = ChatOpenAI(
            model_name=model_name,
            temperature=temperature,
            max_tokens=256,
            api_key=os.getenv('OPENAI_API_KEY')
        )
        self.rewrite_prompt = PromptTemplate(
            input_variables=["original_query"],
            template="""Rewrite the user's query to improve clarity, specificity, and intent understanding. 
Keep the core meaning but make it more explicit for document retrieval.

Original Query: {original_query}

Rewritten Query:"""
        )
    
    def rewrite(self, query: str) -> str:
        """Rewrite a single query for improved clarity."""
        prompt_text = self.rewrite_prompt.format(original_query=query)
        message = HumanMessage(content=prompt_text)
        response = self.llm.invoke([message])
        return response.content
    
    def batch_rewrite(self, queries: List[str]) -> List[str]:
        """Rewrite multiple queries."""
        return [self.rewrite(q) for q in queries]

print('✓ QueryRewriter class defined')

Step 7.2: MULTI-QUERY GENERATION MODULE

In [ ]:

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

class MultiQueryGenerator:
    """Generates 3-5 query variations and merges unique results."""
    
    def __init__(self, model_name: str = "gpt-4-turbo", num_queries: int = 4, temperature: float = 0.7):
        self.llm = ChatOpenAI(
            model_name=model_name,
            temperature=temperature,
            max_tokens=512,
            api_key=os.getenv('OPENAI_API_KEY')
        )
        self.num_queries = num_queries
        self.generation_prompt = PromptTemplate(
            input_variables=["original_query", "num_queries"],
            template="""Generate {num_queries} different variations of the user query to capture different aspects and phrasings.
Each variation should be on a new line, numbered. Keep each query concise and focused.

Original Query: {original_query}

Query Variations:"""
        )
    
    def generate_queries(self, query: str) -> List[str]:
        """Generate multiple query variations."""
        prompt_text = self.generation_prompt.format(
            original_query=query,
            num_queries=self.num_queries
        )
        message = HumanMessage(content=prompt_text)
        response = self.llm.invoke([message])
        
        # Parse response and extract queries
        lines = response.content.split('\n')
        variations = []
        for line in lines:
            # Remove numbering (1., 2., etc.)
            cleaned = line.lstrip('0123456789.-) ').strip()
            if cleaned:
                variations.append(cleaned)
        
        return [query] + variations[:self.num_queries-1]
    
    def merge_results(self, search_results: List[List[Dict]]) -> List[Dict]:
        """Merge and deduplicate results from multiple query searches."""
        merged = {}
        for query_results in search_results:
            for result in query_results:
                vec_id = result.get('id')
                if vec_id not in merged:
                    merged[vec_id] = {
                        **result,
                        'occurrence_count': 1,
                        'combined_score': result.get('score', 0)
                    }
                else:
                    merged[vec_id]['occurrence_count'] += 1
                    merged[vec_id]['combined_score'] += result.get('score', 0)
        
        # Sort by combined score and occurrence
        sorted_results = sorted(
            merged.values(),
            key=lambda x: (x['occurrence_count'], x['combined_score']),
            reverse=True
        )
        return sorted_results

print('✓ MultiQueryGenerator class defined')

Step 7.3: HYDE (HYPOTHETICAL DOCUMENT EMBEDDINGS) MODULE

In [ ]:
# 

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

class HyDEGenerator:
    """Generates hypothetical documents to improve semantic search."""
    
    def __init__(self, llm_model: str = "gpt-4-turbo", embedding_model=None, temperature: float = 0.7):
        self.llm = ChatOpenAI(
            model_name=llm_model,
            temperature=temperature,
            max_tokens=512,
            api_key=os.getenv('OPENAI_API_KEY')
        )
        self.embedding_model = embedding_model or embeddings_model
        self.hyde_prompt = PromptTemplate(
            input_variables=["query"],
            template="""Imagine you are writing a comprehensive financial document that answers the following query.
Write a natural, detailed response that includes relevant financial terminology, numbers, and context.
Keep it focused and 150-250 words.

Query: {query}

Hypothetical Document:"""
        )
    
    def generate_hypothetical_doc(self, query: str) -> str:
        """Generate a hypothetical document for a query."""
        prompt_text = self.hyde_prompt.format(query=query)
        message = HumanMessage(content=prompt_text)
        response = self.llm.invoke([message])
        return response.content
    
    def get_hyde_embedding(self, query: str) -> List[float]:
        """Generate hypothetical document and embed it."""
        hypo_doc = self.generate_hypothetical_doc(query)
        embedding = self.embedding_model.embed_query(hypo_doc)
        return embedding
    
    def batch_hyde_embeddings(self, queries: List[str]) -> List[List[float]]:
        """Generate HyDE embeddings for multiple queries."""
        return [self.get_hyde_embedding(q) for q in queries]

print('✓ HyDEGenerator class defined')

Step 7.4: DOMAIN-AWARE ROUTING MODULE

In [ ]:


from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

class DomainRouter:
    """Classifies queries into domains and selects appropriate retrievers."""
    
    DOMAINS = {
        'finance': {
            'keywords': ['revenue', 'earnings', 'financial', 'profit', 'cash flow', 'balance sheet', 'quarterly', 'annual', 'stock', 'dividend'],
            'sections': ['Item 7', 'Item 8'],  # MD&A, Financial Statements
            'priority_score': 1.0
        },
        'operations': {
            'keywords': ['business', 'operations', 'segment', 'product', 'service', 'market', 'geographic', 'sales'],
            'sections': ['Item 1', 'Item 1A'],  # Business, Risk Factors
            'priority_score': 0.8
        },
        'risk': {
            'keywords': ['risk', 'uncertainty', 'liability', 'threat', 'challenge', 'competition'],
            'sections': ['Item 1A'],  # Risk Factors
            'priority_score': 0.9
        }
    }
    
    def __init__(self, llm_model: str = "gpt-3.5-turbo", temperature: float = 0.1):
        self.llm = ChatOpenAI(
            model_name=llm_model,
            temperature=temperature,
            max_tokens=256,
            api_key=os.getenv('OPENAI_API_KEY')
        )
        self.classification_prompt = PromptTemplate(
            input_variables=["query"],
            template="""Classify this financial query into ONE primary domain: 'finance', 'operations', or 'risk'.
Return only the domain name, nothing else.

Query: {query}

Domain:"""
        )
    
    def classify_domain(self, query: str) -> str:
        """Classify query into a domain using LLM."""
        prompt_text = self.classification_prompt.format(query=query)
        message = HumanMessage(content=prompt_text)
        response = self.llm.invoke([message])
        domain = response.content.lower()
        
        # Validate domain
        if domain not in self.DOMAINS:
            domain = self._fallback_classify(query)
        
        return domain
    
    def _fallback_classify(self, query: str) -> str:
        """Fallback keyword-based classification if LLM fails."""
        query_lower = query.lower()
        scores = {}
        
        for domain, config in self.DOMAINS.items():
            score = sum(1 for kw in config['keywords'] if kw in query_lower)
            scores[domain] = score
        
        return max(scores, key=scores.get) if max(scores.values()) > 0 else 'finance'
    
    def get_section_filters(self, domain: str) -> List[str]:
        """Get relevant sections for classified domain."""
        return self.DOMAINS.get(domain, {}).get('sections', [])
    
    def get_routing_config(self, query: str) -> Dict:
        """Get complete routing configuration for a query."""
        domain = self.classify_domain(query)
        config = self.DOMAINS.get(domain, self.DOMAINS['finance'])
        
        return {
            'domain': domain,
            'sections': config['sections'],
            'priority_score': config['priority_score'],
            'retriever_type': 'hybrid' if domain == 'finance' else 'semantic'
        }

print('✓ DomainRouter class defined')

✓ DomainRouter class defined


Step 7.5: PRE-RETRIEVAL PIPELINE CONTROLLER

In [ ]:
# 

class PreRetrievalPipeline:
    """Orchestrates all pre-retrieval components for optimal search."""
    
    def __init__(
        self,
        embeddings_model,
        index,
        use_query_rewriting: bool = True,
        use_multi_query: bool = True,
        use_hyde: bool = True,
        use_domain_routing: bool = True,
        num_queries: int = 4,
        top_k: int = 5
    ):
        self.embeddings_model = embeddings_model
        self.index = index
        self.top_k = top_k
        
        self.query_rewriter = QueryRewriter() if use_query_rewriting else None
        self.multi_query_gen = MultiQueryGenerator(num_queries=num_queries) if use_multi_query else None
        self.hyde_generator = HyDEGenerator(embedding_model=embeddings_model) if use_hyde else None
        self.domain_router = DomainRouter() if use_domain_routing else None
    
    def process_query(self, user_query: str, verbose: bool = False) -> Dict:
        """Complete pre-retrieval pipeline for a single query."""
        
        pipeline_output = {
            'original_query': user_query,
            'stages': {}
        }
        
        # Stage 1: Query Rewriting
        if self.query_rewriter:
            rewritten = self.query_rewriter.rewrite(user_query)
            pipeline_output['stages']['rewrite'] = {'original': user_query, 'rewritten': rewritten}
            query_for_next = rewritten
        else:
            query_for_next = user_query
        
        # Stage 2: Domain Routing
        if self.domain_router:
            routing_config = self.domain_router.get_routing_config(query_for_next)
            pipeline_output['stages']['routing'] = routing_config
        else:
            routing_config = {'domain': 'all', 'sections': [], 'retriever_type': 'hybrid'}
        
        # Stage 3: Multi-Query or HyDE
        search_strategies = []
        
        if self.multi_query_gen:
            queries = self.multi_query_gen.generate_queries(query_for_next)
            pipeline_output['stages']['multi_query'] = {'variations': queries}
            search_strategies.append(('multi_query', queries))
        
        if self.hyde_generator:
            hyde_embedding = self.hyde_generator.get_hyde_embedding(query_for_next)
            pipeline_output['stages']['hyde'] = {'embedding_dim': len(hyde_embedding)}
            search_strategies.append(('hyde', [hyde_embedding]))
        
        # Stage 4: Unified Search
        all_results = []
        
        # Multi-query search
        if search_strategies and search_strategies[0][0] == 'multi_query':
            for variant in search_strategies[0][1]:
                embedding = self.embeddings_model.embed_query(variant)
                results = self.index.query(
                    vector=embedding,
                    top_k=self.top_k,
                    include_metadata=True
                )
                all_results.append([dict(r) for r in results['matches']])
            
            merged_results = self.multi_query_gen.merge_results(all_results)
        
        # HyDE search
        elif search_strategies and search_strategies[0][0] == 'hyde':
            results = self.index.query(
                vector=search_strategies[0][1][0],
                top_k=self.top_k,
                include_metadata=True
            )
            merged_results = [dict(r) for r in results['matches']]
        
        # Fallback to single query embedding
        else:
            embedding = self.embeddings_model.embed_query(query_for_next)
            results = self.index.query(
                vector=embedding,
                top_k=self.top_k,
                include_metadata=True
            )
            merged_results = [dict(r) for r in results['matches']]
        
        pipeline_output['retrieval_results'] = merged_results
        pipeline_output['metadata'] = {
            'num_results': len(merged_results),
            'domain': routing_config.get('domain'),
            'retriever_type': routing_config.get('retriever_type')
        }
        
        if verbose:
            self._print_pipeline_summary(pipeline_output)
        
        return pipeline_output
    
    def _print_pipeline_summary(self, output: Dict):
        """Print processing summary."""
        print('\n' + '-'*80)
        print('PRE-RETRIEVAL PIPELINE SUMMARY')
        print('-'*80)
        print(f"Original Query: {output['original_query']}")
        
        if 'rewrite' in output['stages']:
            print(f"Rewritten Query: {output['stages']['rewrite']['rewritten']}")
        
        if 'routing' in output['stages']:
            routing = output['stages']['routing']
            print(f"Domain: {routing['domain']} | Retriever: {routing['retriever_type']}")
        
        if 'multi_query' in output['stages']:
            print(f"Query Variations: {len(output['stages']['multi_query']['variations'])} generated")
        
        print(f"\nRetrieved: {output['metadata']['num_results']} documents")
        print('-'*80 + '\n')

print('✓ PreRetrievalPipeline orchestrator defined')

Step 7.6: PIPELINE CONFIGURATION & USAGE EXAMPLES

In [ ]:
# 

print('\n' + '='*80)
print('PIPELINE CONFIGURATION OPTIONS')
print('='*80)

# Configuration Presets

PIPELINE_CONFIGS = {
    'comprehensive': {
        'use_query_rewriting': True,
        'use_multi_query': True,
        'use_hyde': True,
        'use_domain_routing': True,
        'num_queries': 4,
        'top_k': 5
    },
    'balanced': {
        'use_query_rewriting': True,
        'use_multi_query': True,
        'use_hyde': False,
        'use_domain_routing': True,
        'num_queries': 3,
        'top_k': 5
    },
    'fast': {
        'use_query_rewriting': True,
        'use_multi_query': False,
        'use_hyde': False,
        'use_domain_routing': True,
        'num_queries': 1,
        'top_k': 5
    },
    'semantic_focused': {
        'use_query_rewriting': False,
        'use_multi_query': False,
        'use_hyde': True,
        'use_domain_routing': False,
        'num_queries': 1,
        'top_k': 5
    }
}

print('\nAvailable Configurations:')
for config_name, settings in PIPELINE_CONFIGS.items():
    print(f'\n  {config_name.upper()}:')
    print(f'    Query Rewriting: {settings["use_query_rewriting"]}')
    print(f'    Multi-Query: {settings["use_multi_query"]} (variations: {settings["num_queries"]})')
    print(f'    HyDE: {settings["use_hyde"]}')
    print(f'    Domain Routing: {settings["use_domain_routing"]}')

print('\n' + '='*80)
print('USAGE: Create pipeline instance with desired configuration')
print('='*80)
print('\nExample 1 - Comprehensive (best quality):\n')
print('  pipeline = PreRetrievalPipeline(')
print('      embeddings_model=embeddings_model,')
print('      index=index,')
print('      **PIPELINE_CONFIGS["comprehensive"]')
print('  )')
print('  results = pipeline.process_query("What is Apple\'s revenue growth?", verbose=True)')

print('\nExample 2 - Balanced (quality vs speed):\n')
print('  pipeline = PreRetrievalPipeline(')
print('      embeddings_model=embeddings_model,')
print('      index=index,')
print('      **PIPELINE_CONFIGS["balanced"]')
print('  )')
print('  results = pipeline.process_query("Explain risk factors", verbose=True)')

print('\nExample 3 - Custom configuration:\n')
print('  pipeline = PreRetrievalPipeline(')
print('      embeddings_model=embeddings_model,')
print('      index=index,')
print('      use_query_rewriting=True,')
print('      use_multi_query=True,')
print('      use_hyde=False,')
print('      use_domain_routing=True,')
print('      num_queries=3,')
print('      top_k=10')
print('  )')

print('\n' + '='*80)
print('PHASE 6 COMPLETE: PRE-RETRIEVAL PIPELINE READY')
print('='*80)

# Phase 8 : During Retrieval

Step 8.1: HYBRID RETRIEVAL (DENSE + BM25)

In [ ]:
# 

import numpy as np

class HybridRetriever:
    """Combines dense (semantic) and sparse (BM25) retrieval with adjustable weighting."""
    
    def __init__(
        self,
        embeddings_model,
        sparse_generator,
        index,
        alpha: float = 0.5,
        top_k: int = 10
    ):
        """
        Args:
            embeddings_model: OpenAI embeddings model
            sparse_generator: BM25SparseVectorGenerator instance
            index: Pinecone index
            alpha: Weight for dense retrieval (0.0-1.0), sparse weight = 1-alpha
            top_k: Number of results to retrieve
        """
        self.embeddings_model = embeddings_model
        self.sparse_generator = sparse_generator
        self.index = index
        self.alpha = alpha
        self.top_k = top_k
    
    def _normalize_scores(self, scores: List[float]) -> List[float]:
        """Normalize scores to [0, 1] range."""
        if not scores:
            return scores
        min_score = min(scores)
        max_score = max(scores)
        if max_score - min_score == 0:
            return [0.5] * len(scores)
        return [(s - min_score) / (max_score - min_score) for s in scores]
    
    def retrieve(self, query: str) -> List[Dict]:
        """Hybrid retrieval combining dense and sparse search."""
        dense_results = self._dense_retrieve(query)
        sparse_results = self._sparse_retrieve(query)
        
        return self._merge_results(dense_results, sparse_results)
    
    def _dense_retrieve(self, query: str) -> List[Dict]:
        """Dense retrieval using OpenAI embeddings."""
        embedding = self.embeddings_model.embed_query(query)
        results = self.index.query(
            vector=embedding,
            top_k=self.top_k * 2,
            include_metadata=True
        )
        
        return [
            {
                'id': r['id'],
                'score': r['score'],
                'metadata': r.get('metadata', {}),
                'source': 'dense'
            }
            for r in results['matches']
        ]
    
    def _sparse_retrieve(self, query: str) -> List[Dict]:
        """Sparse retrieval using BM25."""
        tokens = self.sparse_generator._tokenize(query)
        
        sparse_scores = {}
        for token in tokens:
            if token in self.sparse_generator.token_to_idx:
                token_idx = self.sparse_generator.token_to_idx[token]
                freq = tokens.count(token)
                score = freq / (len(tokens) + 1e-8)
                sparse_scores[token_idx] = min(score, 1.0)
        
        return [{
            'id': None,
            'score': score,
            'sparse_scores': sparse_scores,
            'source': 'sparse'
        }]
    
    def _merge_results(self, dense_results: List[Dict], sparse_results: List[Dict]) -> List[Dict]:
        """Merge dense and sparse results with alpha weighting."""
        merged = {}
        
        dense_scores = [r['score'] for r in dense_results]
        normalized_dense = self._normalize_scores(dense_scores)
        
        for result, norm_score in zip(dense_results, normalized_dense):
            doc_id = result['id']
            merged[doc_id] = {
                **result,
                'hybrid_score': self.alpha * norm_score,
                'dense_score': norm_score,
                'sparse_score': 0.0
            }
        
        sorted_results = sorted(
            merged.values(),
            key=lambda x: x['hybrid_score'],
            reverse=True
        )
        
        return sorted_results[:self.top_k]

print('✓ HybridRetriever class defined')

✓ HybridRetriever class defined


Step 8.2: MAXIMAL MARGINAL RELEVANCE (MMR) RERANKING

In [ ]:
# 

class MMRReranker:
    """Removes redundant results and ensures diversity using Maximal Marginal Relevance."""
    
    def __init__(self, embeddings_model, lambda_param: float = 0.5):
        """
        Args:
            embeddings_model: Embeddings model for calculating similarity
            lambda_param: Balance between relevance (1.0) and diversity (0.0)
        """
        self.embeddings_model = embeddings_model
        self.lambda_param = lambda_param
    
    def _cosine_similarity(self, vec1: List[float], vec2: List[float]) -> float:
        """Calculate cosine similarity between two vectors."""
        vec1 = np.array(vec1)
        vec2 = np.array(vec2)
        norm_product = np.linalg.norm(vec1) * np.linalg.norm(vec2)
        if norm_product == 0:
            return 0.0
        return np.dot(vec1, vec2) / norm_product
    
    def rerank(
        self,
        query: str,
        documents: List[Dict],
        top_k: int = 5
    ) -> List[Dict]:
        """Apply MMR to ensure diversity in top-k results."""
        if not documents:
            return []
        
        query_embedding = self.embeddings_model.embed_query(query)
        
        selected = []
        remaining = list(documents)
        doc_embeddings = {}
        
        while len(selected) < top_k and remaining:
            mmr_scores = []
            
            for doc in remaining:
                doc_id = doc['id']
                
                if doc_id not in doc_embeddings:
                    content = doc.get('metadata', {}).get('text_preview', '')
                    if content:
                        doc_embeddings[doc_id] = self.embeddings_model.embed_query(content)
                
                relevance = doc.get('hybrid_score', 0.0)
                
                diversity = 1.0
                if selected:
                    selected_sims = [
                        self._cosine_similarity(
                            doc_embeddings[doc_id],
                            doc_embeddings[s['id']]
                        )
                        for s in selected
                        if s['id'] in doc_embeddings
                    ]
                    if selected_sims:
                        diversity = 1.0 - max(selected_sims)
                
                mmr_score = self.lambda_param * relevance + (1 - self.lambda_param) * diversity
                mmr_scores.append((mmr_score, doc))
            
            if mmr_scores:
                best_doc = max(mmr_scores, key=lambda x: x[0])[1]
                selected.append({**best_doc, 'mmr_score': mmr_scores[0][0]})
                remaining.remove(best_doc)
            else:
                break
        
        return selected

print('✓ MMRReranker class defined')

✓ MMRReranker class defined


Step 8.3: CROSS-ENCODER RERANKING

In [ ]:
# 

class CrossEncoderReranker:
    """Reranks documents using a cross-encoder model (BGE or Cohere)."""
    
    def __init__(self, reranker_type: str = "bgm25"):
        """
        Args:
            reranker_type: 'bge' for BGERerank or 'cohere' for Cohere API
        """
        self.reranker_type = reranker_type
        self.reranker = None
        self._init_reranker()
    
    def _init_reranker(self):
        """Initialize the reranker model."""
        if self.reranker_type == "bge":
            try:
                from langchain_core.retrievers import ContextualCompressionRetriever
                from langchain_community.document_compressors import CrossEncoderRecompressor
                from langchain_community.cross_encoders import HuggingFaceCrossEncoder
                
                model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
                self.reranker = model
                print('✓ BGE cross-encoder initialized')
            except ImportError:
                print('⚠ BGE cross-encoder requires: pip install sentence-transformers')
                self.reranker = None
        
        elif self.reranker_type == "cohere":
            try:
                import cohere
                self.reranker = cohere.Client(api_key=os.getenv('COHERE_API_KEY'))
                print('✓ Cohere reranker initialized')
            except ImportError:
                print('⚠ Cohere reranker requires: pip install cohere')
                self.reranker = None
    
    def rerank(
        self,
        query: str,
        documents: List[Dict],
        top_k: int = 3
    ) -> List[Dict]:
        """Rerank documents using cross-encoder model."""
        if not documents or not self.reranker:
            return documents[:top_k]
        
        if self.reranker_type == "bge":
            return self._rerank_with_bge(query, documents, top_k)
        elif self.reranker_type == "cohere":
            return self._rerank_with_cohere(query, documents, top_k)
        else:
            return documents[:top_k]
    
    def _rerank_with_bge(
        self,
        query: str,
        documents: List[Dict],
        top_k: int
    ) -> List[Dict]:
        """BGE cross-encoder reranking."""
        doc_texts = [
            d.get('metadata', {}).get('text_preview', '')
            for d in documents
        ]
        
        scores = self.reranker.predict([[query, doc] for doc in doc_texts])
        
        scored_docs = []
        for doc, score in zip(documents, scores):
            doc_copy = dict(doc)
            doc_copy['reranker_score'] = float(score)
            scored_docs.append(doc_copy)
        
        sorted_docs = sorted(scored_docs, key=lambda x: x['reranker_score'], reverse=True)
        return sorted_docs[:top_k]
    
    def _rerank_with_cohere(
        self,
        query: str,
        documents: List[Dict],
        top_k: int
    ) -> List[Dict]:
        """Cohere API-based reranking."""
        doc_texts = [
            d.get('metadata', {}).get('text_preview', '')
            for d in documents
        ]
        
        results = self.reranker.rerank(
            model="rerank-english-v2.0",
            query=query,
            documents=doc_texts,
            top_n=top_k
        )
        
        reranked_docs = []
        for result in results:
            doc = documents[result.index]
            doc_copy = dict(doc)
            doc_copy['reranker_score'] = result.relevance_score
            reranked_docs.append(doc_copy)
        
        return reranked_docs

print('✓ CrossEncoderReranker class defined')

✓ CrossEncoderReranker class defined


Step 8.4: DURING-RETRIEVAL PIPELINE ORCHESTRATOR

In [ ]:
# 

class DuringRetrievalPipeline:
    """Orchestrates complete retrieval pipeline: hybrid search → MMR → reranking."""
    
    def __init__(
        self,
        embeddings_model,
        sparse_generator,
        index,
        use_hybrid: bool = True,
        use_mmr: bool = True,
        use_reranking: bool = True,
        reranker_type: str = "bge",
        alpha: float = 0.5,
        lambda_param: float = 0.5,
        top_k: int = 5,
        mmr_k: int = 10
    ):
        """
        Args:
            embeddings_model: OpenAI embeddings
            sparse_generator: BM25 generator
            index: Pinecone index
            use_hybrid: Enable hybrid retrieval
            use_mmr: Enable MMR reranking
            use_reranking: Enable cross-encoder reranking
            reranker_type: 'bge' or 'cohere'
            alpha: Dense vs sparse weight (0.0-1.0)
            lambda_param: Relevance vs diversity for MMR
            top_k: Final number of results
            mmr_k: Intermediate MMR results
        """
        self.embeddings_model = embeddings_model
        self.sparse_generator = sparse_generator
        self.index = index
        self.top_k = top_k
        
        self.hybrid_retriever = HybridRetriever(
            embeddings_model, sparse_generator, index, alpha=alpha
        ) if use_hybrid else None
        
        self.mmr_reranker = MMRReranker(embeddings_model, lambda_param) if use_mmr else None
        
        self.cross_encoder = CrossEncoderReranker(reranker_type) if use_reranking else None
        
        self.use_hybrid = use_hybrid
        self.use_mmr = use_mmr
        self.use_reranking = use_reranking
        self.mmr_k = mmr_k
    
    def retrieve(self, query: str, verbose: bool = False) -> Dict:
        """Complete retrieval pipeline."""
        pipeline_output = {
            'query': query,
            'stages': {}
        }
        
        if verbose:
            print('\n' + '='*80)
            print(f'DURING-RETRIEVAL PIPELINE: "{query}"')
            print('='*80)
        
        # Stage 1: Hybrid Retrieval
        if self.hybrid_retriever:
            hybrid_results = self.hybrid_retriever.retrieve(query)
            pipeline_output['stages']['hybrid'] = {
                'count': len(hybrid_results),
                'results': hybrid_results
            }
            if verbose:
                print(f'\n[Stage 1] Hybrid Retrieval')
                print(f'  Dense + BM25 retrieval: {len(hybrid_results)} documents')
            current_results = hybrid_results
        else:
            embedding = self.embeddings_model.embed_query(query)
            results = self.index.query(vector=embedding, top_k=self.top_k, include_metadata=True)
            current_results = [dict(r) for r in results['matches']]
            if verbose:
                print(f'\n[Stage 1] Dense Retrieval Only: {len(current_results)} documents')
        
        # Stage 2: MMR Reranking
        if self.mmr_reranker and len(current_results) > 0:
            mmr_results = self.mmr_reranker.rerank(query, current_results, self.mmr_k)
            pipeline_output['stages']['mmr'] = {
                'count': len(mmr_results),
                'diversity_lambda': self.mmr_reranker.lambda_param
            }
            if verbose:
                print(f'\n[Stage 2] MMR Reranking')
                print(f'  Removed redundancy: {len(current_results)} → {len(mmr_results)} documents')
            current_results = mmr_results
        
        # Stage 3: Cross-Encoder Reranking
        if self.cross_encoder and len(current_results) > 0:
            final_results = self.cross_encoder.rerank(query, current_results, self.top_k)
            pipeline_output['stages']['cross_encoder'] = {
                'count': len(final_results),
                'model': self.cross_encoder.reranker_type
            }
            if verbose:
                print(f'\n[Stage 3] Cross-Encoder Reranking')
                print(f'  Fine-tuned ranking: {len(current_results)} → {len(final_results)} documents')
            current_results = final_results
        else:
            current_results = current_results[:self.top_k]
        
        pipeline_output['final_results'] = current_results
        pipeline_output['total_results'] = len(current_results)
        
        if verbose:
            self._print_summary(pipeline_output)
        
        return pipeline_output
    
    def _print_summary(self, output: Dict):
        """Print retrieval pipeline summary."""
        print('\n' + '-'*80)
        print('RETRIEVAL RESULTS')
        print('-'*80)
        
        for i, doc in enumerate(output['final_results'], 1):
            print(f'\n[{i}] Document')
            print(f'  ID: {doc.get("id", "N/A")}')
            if 'hybrid_score' in doc:
                print(f'  Hybrid Score: {doc["hybrid_score"]:.4f}')
            if 'mmr_score' in doc:
                print(f'  MMR Score: {doc["mmr_score"]:.4f}')
            if 'reranker_score' in doc:
                print(f'  Reranker Score: {doc["reranker_score"]:.4f}')
            
            metadata = doc.get('metadata', {})
            print(f'  Section: {metadata.get("section", "N/A")}')
            print(f'  Company: {metadata.get("company", "N/A")}')
        
        print('\n' + '-'*80)

print('✓ DuringRetrievalPipeline orchestrator defined')

Step 8.5: DURING-RETRIEVAL PIPELINE CONFIGURATION & USAGE

In [ ]:
# 
DURING_RETRIEVAL_CONFIGS = {
    'comprehensive': {
        'use_hybrid': True,
        'use_mmr': True,
        'use_reranking': True,
        'reranker_type': 'bge',
        'alpha': 0.5,
        'lambda_param': 0.5,
        'top_k': 5,
        'mmr_k': 10
    },
    'fast': {
        'use_hybrid': True,
        'use_mmr': False,
        'use_reranking': False,
        'alpha': 0.6,
        'top_k': 5
    },
    'diversity_focused': {
        'use_hybrid': True,
        'use_mmr': True,
        'use_reranking': True,
        'reranker_type': 'bge',
        'alpha': 0.5,
        'lambda_param': 0.7,
        'top_k': 5,
        'mmr_k': 15
    },
    'semantic_only': {
        'use_hybrid': False,
        'use_mmr': True,
        'use_reranking': True,
        'reranker_type': 'bge',
        'top_k': 5,
        'mmr_k': 10
    }
}

print('\nAvailable Configurations:')
for config_name, settings in DURING_RETRIEVAL_CONFIGS.items():
    print(f'\n  {config_name.upper()}:')
    print(f'    Hybrid: {settings.get("use_hybrid", False)}')
    print(f'    MMR: {settings.get("use_mmr", False)}')
    print(f'    Reranking: {settings.get("use_reranking", False)}')
    if settings.get("use_hybrid"):
        print(f'    Dense/Sparse weight (alpha): {settings.get("alpha", 0.6)}')
    if settings.get("use_mmr"):
        print(f'    MMR lambda: {settings.get("lambda_param", 0.5)}')

print('\n' + '='*80)
print('USAGE EXAMPLES')
print('='*80)

print('\nExample 1 - Comprehensive Retrieval:\n')
print('  during_pipeline = DuringRetrievalPipeline(')
print('      embeddings_model=embeddings_model,')
print('      sparse_generator=sparse_generator,')
print('      index=index,')
print('      **DURING_RETRIEVAL_CONFIGS["comprehensive"]')
print('  )')
print('  results = during_pipeline.retrieve("Apple revenue growth", verbose=True)')
print('  final_docs = results["final_results"]')

print('\nExample 2 - Fast Retrieval:\n')
print('  during_pipeline = DuringRetrievalPipeline(')
print('      embeddings_model=embeddings_model,')
print('      sparse_generator=sparse_generator,')
print('      index=index,')
print('      **DURING_RETRIEVAL_CONFIGS["fast"]')
print('  )')
print('  results = during_pipeline.retrieve("Financial statement", verbose=False)')

print('\nExample 3 - Diversity-Focused:\n')
print('  during_pipeline = DuringRetrievalPipeline(')
print('      embeddings_model=embeddings_model,')
print('      sparse_generator=sparse_generator,')
print('      index=index,')
print('      **DURING_RETRIEVAL_CONFIGS["diversity_focused"]')
print('  )')
print('  results = during_pipeline.retrieve("Risk factors", verbose=True)')

print('\n' + '='*80)
print('COMPLETE PIPELINE FLOW')
print('='*80)
print('''
Dense Embedding
     ↓
Hybrid Retrieval ←→ BM25 Sparse Vector
     ↓ (10-20 docs)
MMR Reranking (Remove Redundancy)
     ↓ (5-10 diverse docs)
Cross-Encoder Reranking (Fine-tune Ranking)
     ↓ (Top-5 final results)
Final Documents
''')

print('\n' + '='*80)
print('PHASE 7 COMPLETE: DURING-RETRIEVAL PIPELINE READY')
print('='*80)

# phase 9 : post Retriever 

Step 9.1: TOKEN COUNTER UTILITY

In [ ]:
# 

import re
import tiktoken
from typing import List,Dict

class TokenCounter:
    """Estimates and counts tokens using tiktoken."""
    
    def __init__(self, model: str = "gpt-4"):
        """Initialize token counter for specific model."""
        self.model = model
        try:
            self.encoding = tiktoken.encoding_for_model(model)
        except:
            self.encoding = tiktoken.get_encoding("cl100k_base")
    
    def count_text(self, text: str) -> int:
        """Count tokens in text."""
        return len(self.encoding.encode(text))
    
    def count_messages(self, messages: List[Dict]) -> int:
        """Count tokens in message list."""
        total = 0
        for msg in messages:
            total += 4
            for key, value in msg.items():
                total += len(self.encoding.encode(value))
        return total
    
    def estimate_doc_tokens(self, doc: Dict) -> int:
        """Estimate tokens in document."""
        content = doc.get('metadata', {}).get('text_preview', '')
        if not content:
            content = str(doc.get('page_content', ''))
        return self.count_text(content)

print('✓ TokenCounter utility defined')

Step 9.2: CONTEXTUAL COMPRESSION

In [ ]:

class ContextualCompressor:
    """Removes irrelevant content from documents using LLM-based extraction."""
    
    def __init__(self, llm_model: str = "gpt-3.5-turbo", compression_ratio: float = 0.5):
        """
        Args:
            llm_model: LLM model for extraction
            compression_ratio: Target compression (0.0-1.0, lower = more aggressive)
        """
        self.llm = ChatOpenAI(
            model_name=llm_model,
            temperature=0.0,
            max_tokens=512,
            api_key=os.getenv('OPENAI_API_KEY')
        )
        self.compression_ratio = compression_ratio
        self.compression_prompt = PromptTemplate(
            input_variables=["document", "query"],
            template="""Extract only the sentences from the document that directly answer or relate to the query.
Keep the extracted sentences coherent and in order. Remove filler, repetitions, and irrelevant details.

Query: {query}

Document:
{document}

Extracted Relevant Sentences:"""
        )
    
    def compress(self, query: str, document: str) -> str:
        """Extract query-relevant content from document."""
        if len(document) < 100:
            return document
        
        prompt_text = self.compression_prompt.format(query=query, document=document)
        message = HumanMessage(content=prompt_text)
        response = self.llm.invoke([message])
        compressed = response.content.strip()
        
        if not compressed or len(compressed) < 20:
            sentences = re.split(r'[.!?]+', document)
            ratio = max(1, int(len(sentences) * self.compression_ratio))
            compressed = '. '.join(sentences[:ratio])
        
        return compressed
    
    def compress_documents(self, query: str, documents: List[Dict]) -> List[Dict]:
        """Compress multiple documents."""
        compressed = []
        
        for i, doc in enumerate(documents):
            content = doc.get('metadata', {}).get('text_preview', '')
            if not content:
                content = doc.get('page_content', '')
            
            if content:
                compressed_content = self.compress(query, content)
            else:
                compressed_content = content
            
            doc_copy = dict(doc)
            doc_copy['compressed_content'] = compressed_content
            doc_copy['original_length'] = len(content)
            doc_copy['compressed_length'] = len(compressed_content)
            doc_copy['compression_ratio'] = (
                len(compressed_content) / len(content) if content else 0.0
            )
            compressed.append(doc_copy)
        
        return compressed

print('✓ ContextualCompressor class defined')

# Phase 10 : argumentation and response generation

Step 10.1: TEMPLATE DESIGN

In [ ]:
# 

class PromptTemplateBuilder:
    """Constructs system and user prompts with anti-hallucination constraints."""
    
    SYSTEM_PROMPT = """You are a financial Q&A assistant powered by knowledge base retrieval.

STRICT RULES:
1. Only answer based on provided context documents
2. If context doesn't contain answer, respond: "I don't have this information in my knowledge base"
3. Always cite sources with [Doc X: Title] format
4. For multi-part questions, address each part separately
5. Keep responses concise and factual
6. Do not speculate, infer, or provide external knowledge
7. If query is out-of-scope or harmful, decline politely"""
    
    FEW_SHOT_EXAMPLES = [
        {
            "query": "What was Apple's revenue in 2023?",
            "reasoning": "1. Search for Apple financial data 2023\n2. Locate revenue line item\n3. Extract exact figure with source",
            "answer": "Apple reported total revenue of $383.3 billion for fiscal year 2023, ending September 30, 2023. [Doc 1: Apple 10-K 2023]"
        },
        {
            "query": "How does Tesla compare to Ford in market cap?",
            "reasoning": "1. Find Tesla market cap\n2. Find Ford market cap\n3. Calculate comparison ratio\n4. Note data timeliness",
            "answer": "Based on latest available data: [Doc 5: Market Comparisons Q4 2023]. For current market cap, please check real-time financial sources."
        }
    ]
    
    @staticmethod
    def build_system_prompt():
        """Get system prompt."""
        return PromptTemplateBuilder.SYSTEM_PROMPT
    
    @staticmethod
    def build_user_prompt(query: str, context: str, memory_summary: str = "") -> str:
        """Build complete user prompt with context and memory."""
        prompt = f"""Conversation History Summary:
{memory_summary if memory_summary else "No previous conversation"}

Retrieved Context:
{context}

Current Question: {query}

Step-by-step reasoning:
1. Review context documents
2. Identify relevant information
3. Check if all parts are covered in context
4. Formulate answer with citations

Answer:"""
        return prompt
    
    @staticmethod
    def get_few_shot_examples():
        """Get few-shot examples for in-context learning."""
        return PromptTemplateBuilder.FEW_SHOT_EXAMPLES

print('✓ PromptTemplateBuilder class defined')

step 10.2 : NVERSATION MEMORY MANAGER

In [ ]:
# 

import json
import uuid
from datetime import datetime

class ConversationMemoryManager:
    """Manages conversation history with S3 persistence."""
    
    def __init__(self, s3_client, bucket_name: str, max_window: int = 5):
        """
        Args:
            s3_client: boto3 S3 client
            bucket_name: S3 bucket for chat history
            max_window: Number of recent interactions to keep in memory
        """
        self.s3_client = s3_client
        self.bucket_name = bucket_name
        self.max_window = max_window
        self.session_id = str(uuid.uuid4())
        self.conversation_buffer = []
        self.summary = ""
    
    def add_interaction(self, query: str, response: str, sources: List[str] = None):
        """Add query-response pair to buffer."""
        interaction = {
            "timestamp": datetime.utcnow().isoformat(),
            "query": query,
            "response": response,
            "sources": sources or []
        }
        self.conversation_buffer.append(interaction)
        
        if len(self.conversation_buffer) > self.max_window:
            self.conversation_buffer.pop(0)
    
    def update_summary(self, summary_text: str):
        """Update conversation summary."""
        self.summary = summary_text
    
    def get_memory_string(self) -> str:
        """Get formatted memory for prompt injection."""
        if not self.summary and not self.conversation_buffer:
            return ""
        
        memory_parts = []
        
        if self.summary:
            memory_parts.append(f"Previous Topics: {self.summary}")
        
        if self.conversation_buffer:
            recent = "Recent Questions:\n"
            for i, interaction in enumerate(self.conversation_buffer[-3:], 1):
                recent += f"{i}. Q: {interaction['query'][:100]}...\n"
            memory_parts.append(recent)
        
        return "\n".join(memory_parts)
    
    def save_to_s3(self) -> bool:
        """Save full session to S3."""
        session_data = {
            "session_id": self.session_id,
            "created_at": datetime.utcnow().isoformat(),
            "summary": self.summary,
            "interactions": self.conversation_buffer
        }
        
        try:
            key = f"chat-history/{self.session_id}/session.json"
            self.s3_client.put_object(
                Bucket=self.bucket_name,
                Key=key,
                Body=json.dumps(session_data, indent=2),
                ContentType="application/json"
            )
            return True
        except Exception as e:
            print(f"Error saving to S3: {e}")
            return False
    
    def load_session_from_s3(self, session_id: str) -> bool:
        """Load previous session from S3."""
        try:
            key = f"chat-history/{session_id}/session.json"
            response = self.s3_client.get_object(Bucket=self.bucket_name, Key=key)
            session_data = json.loads(response['Body'].read().decode('utf-8'))
            
            self.session_id = session_data["session_id"]
            self.summary = session_data.get("summary", "")
            self.conversation_buffer = session_data.get("interactions", [])[-self.max_window:]
            return True
        except Exception as e:
            print(f"Error loading session: {e}")
            return False

print('✓ ConversationMemoryManager class defined')

Step 10.3: CHAIN-OF-THOUGHT REASONING

In [ ]:
# 

class ChainOfThoughtReasoner:
    """Generates step-by-step reasoning for complex queries."""
    
    @staticmethod
    def analyze_query_complexity(query: str) -> Dict:
        """Determine if query requires chain-of-thought reasoning."""
        complexity_indicators = {
            "multi_part": len([q for q in query.split("?") if q.strip()]) > 1,
            "comparison": any(word in query.lower() for word in ["compare", "vs", "versus", "difference"]),
            "calculation": any(word in query.lower() for word in ["calculate", "how much", "percent", "ratio"]),
            "explanation": any(word in query.lower() for word in ["explain", "why", "how does", "mechanism"]),
            "historical": any(word in query.lower() for word in ["trend", "change", "history", "evolution"])
        }
        
        requires_cot = any(complexity_indicators.values())
        return {
            "requires_cot": requires_cot,
            "indicators": complexity_indicators,
            "complexity_score": sum(complexity_indicators.values())
        }
    
    @staticmethod
    def build_cot_prompt(query: str, context: str) -> str:
        """Build chain-of-thought prompt for complex reasoning."""
        cot_template = """Reasoning Steps:

1. Extract Key Information:
   - What are the main entities/concepts in the query?
   - What information is explicitly provided in context?

2. Identify Relationships:
   - How do the concepts relate to each other?
   - What connections exist in the provided context?

3. Synthesize Answer:
   - Combine information from multiple parts if needed
   - Build logical chain from retrieved facts

4. Validate Against Context:
   - Is every claim backed by context?
   - Are all sources cited?

Based on above reasoning, provide your answer:"""
        return cot_template
    
    @staticmethod
    def extract_reasoning_steps(llm_response: str) -> Dict:
        """Parse reasoning steps from LLM response."""
        steps = []
        lines = llm_response.split("\n")
        
        for line in lines:
            if line.strip().startswith(("-", "•", "*")):
                steps.append(line.strip())
        
        return {
            "reasoning_steps": steps,
            "step_count": len(steps)
        }

print('✓ ChainOfThoughtReasoner class defined')

Step 10.4: RAG GENERATION PIPELINE

In [ ]:


class RAGGenerationPipeline:
    """Complete pipeline: query → memory → prompt → LLM → answer."""
    
    def __init__(self, llm_client, s3_client: str, bucket_name: str):
        """
        Args:
            llm_client: OpenAI or Claude client
            s3_client: boto3 S3 client
            bucket_name: S3 bucket for chat history
        """
        self.llm_client = llm_client
        self.memory_manager = ConversationMemoryManager(s3_client, bucket_name)
        self.prompt_builder = PromptTemplateBuilder()
        self.reasoner = ChainOfThoughtReasoner()
    
    def generate_response(
        self,
        query: str,
        retrieved_documents: List[Dict],
        session_id: str = None,
        use_cot: bool = True
    ) -> Dict:
        """Execute full generation pipeline."""
        
        # Load previous session if provided
        if session_id:
            self.memory_manager.load_session_from_s3(session_id)
        
        # Build context from retrieved documents
        context = self._format_context(retrieved_documents)
        
        # Get conversation memory
        memory_summary = self.memory_manager.get_memory_string()
        
        # Analyze query complexity
        complexity = self.reasoner.analyze_query_complexity(query)
        
        # Build prompt
        system_prompt = self.prompt_builder.build_system_prompt()
        user_prompt = self.prompt_builder.build_user_prompt(query, context, memory_summary)
        
        # Add chain-of-thought if needed
        if use_cot and complexity["requires_cot"]:
            user_prompt += "\n\n" + self.reasoner.build_cot_prompt(query, context)
        
        # Add few-shot examples
        user_prompt += self._format_few_shot_examples(complexity["indicators"])
        
        # Call LLM
        response_text = self._call_llm(system_prompt, user_prompt)
        
        # Extract citations
        citations = self._extract_citations(response_text, retrieved_documents)
        
        # Store in memory
        sources = [doc.get("metadata", {}).get("source", "Unknown") for doc in retrieved_documents[:3]]
        self.memory_manager.add_interaction(query, response_text, sources)
        
        # Save session
        self.memory_manager.save_to_s3()
        
        return {
            "query": query,
            "response": response_text,
            "citations": citations,
            "session_id": self.memory_manager.session_id,
            "used_cot": (use_cot and complexity["requires_cot"]),
            "complexity_score": complexity["complexity_score"]
        }
    
    def _format_context(self, documents: List[Dict]) -> str:
        """Format retrieved documents into context string."""
        context_parts = []
        
        for i, doc in enumerate(documents[:5], 1):
            title = doc.get("metadata", {}).get("title", "Document")
            source = doc.get("metadata", {}).get("source", "Unknown")
            content = doc.get("metadata", {}).get("text_preview", "")
            
            context_parts.append(
                f"[Doc {i}: {title} (Source: {source})]\n{content}\n"
            )
        
        return "\n".join(context_parts) if context_parts else "No relevant documents found."
    
    def _format_few_shot_examples(self, indicators: Dict) -> str:
        """Include relevant few-shot examples based on query type."""
        examples = self.prompt_builder.get_few_shot_examples()
        
        if indicators.get("comparison") and len(examples) > 1:
            example = examples[1]
            return f"\nExample for comparison queries:\nQ: {example['query']}\nA: {example['answer']}"
        elif len(examples) > 0:
            example = examples[0]
            return f"\nExample format:\nQ: {example['query']}\nA: {example['answer']}"
        
        return ""
    
    def _call_llm(self, system_prompt: str, user_prompt: str) -> str:
        """Call LLM with prompts (to be implemented with actual API)."""
        try:
            completion = self.llm_client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.3,
                max_tokens=500
            )
            return completion.choices[0].message.content
        except Exception as e:
            return f"Error generating response: {str(e)}"
    
    def _extract_citations(self, response: str, documents: List[Dict]) -> List[Dict]:
        """Extract document citations from response."""
        citations = []
        
        for doc in documents[:3]:
            title = doc.get("metadata", {}).get("title", "Document")
            source = doc.get("metadata", {}).get("source", "Unknown")
            
            citations.append({
                "title": title,
                "source": source,
                "relevance": doc.get("score", 0.0)
            })
        
        return citations

print('✓ RAGGenerationPipeline class defined')